In [ ]:
# %% [markdown]
# # Zero-Model Intelligence: Visual Policy Map Animation
# 
# This notebook creates an animated demonstration of how Visual Policy Maps transform raw policy evaluation data into spatially-organized decision maps where intelligence is embedded in the data structure itself.
# 
# The animation shows:
# 1. Raw, unsorted policy data (documents × metrics)
# 2. Task-aware sorting of documents and metrics
# 3. Focusing on the most relevant region (top-left corner)
# 4. Edge device decision-making by reading just one pixel
# 5. Hierarchical zooming to show multi-level decision processing

# %% [code]
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.gridspec as gridspec
from IPython.display import Image, display
import os
import time

# Configuration
NUM_DOCUMENTS = 80      # Number of policy documents
NUM_METRICS = 40        # Number of metrics per document
ZOOM_FACTOR = 3         # Zoom factor at each hierarchical level
NUM_LEVELS = 3          # Number of hierarchical levels to demonstrate
FRAMES_PER_STEP = 30    # Frames for each animation step
FPS = 20                # Frames per second for the output GIF
TASK = "uncertainty"    # Starting task for demonstration

# %% [code]
def generate_policy_data(num_docs, num_metrics):
    """Generate realistic synthetic policy evaluation data with boundary checks"""
    data = np.zeros((num_docs, num_metrics))
    
    # Create realistic score distributions with patterns
    # Uncertainty: higher for early documents
    data[:, 0] = np.linspace(0.9, 0.1, num_docs)
    
    # Size: random but correlated with uncertainty (if we have at least 2 metrics)
    if num_metrics > 1:
        data[:, 1] = 0.5 + 0.5 * np.random.rand(num_docs) - 0.3 * data[:, 0]
    
    # Quality: higher for later documents (if we have at least 3 metrics)
    if num_metrics > 2:
        data[:, 2] = np.linspace(0.2, 0.9, num_docs)
    
    # Novelty: random (if we have at least 4 metrics)
    if num_metrics > 3:
        data[:, 3] = np.random.rand(num_docs)
    
    # Coherence: correlated with quality (if we have at least 5 metrics)
    if num_metrics > 4:
        data[:, 4] = data[:, 2] * 0.7 + 0.3 * np.random.rand(num_docs)
    
    # Fill remaining metrics with random values
    for i in range(5, num_metrics):
        data[:, i] = np.random.rand(num_docs)
    
    # Ensure values are in [0,1] range
    data = np.clip(data, 0, 1)
    
    return data

# %% [code]
def sort_policy_data(data, task="uncertainty"):
    """
    Sort policy data based on task relevance with boundary checks.
    
    Args:
         Policy evaluation data (documents × metrics)
        task: Task description for sorting
    
    Returns:
        sorted_data: Data sorted by document relevance and metric importance
        metric_order: Order of metrics after sorting
        doc_order: Order of documents after sorting
    """
    num_docs, num_metrics = data.shape
    
    # Define task-specific importance weights with safe access
    task_weights = {
        "uncertainty": {
            "uncertainty": 1.0, "size": 0.7, "quality": 0.2, 
            "novelty": 0.3, "coherence": 0.1
        },
        "quality": {
            "quality": 1.0, "coherence": 0.8, "novelty": 0.5,
            "uncertainty": 0.2, "size": 0.1
        },
        "size": {
            "size": 1.0, "uncertainty": 0.6, "quality": 0.3,
            "novelty": 0.2, "coherence": 0.1
        }
    }
    
    # Get weights for current task
    weights = task_weights.get(task, task_weights["uncertainty"])
    
    # Sort metrics by task importance
    metric_importance = np.zeros(num_metrics)
    
    # Only set weights for metrics that exist
    metric_names = ["uncertainty", "size", "quality", "novelty", "coherence"]
    for i, name in enumerate(metric_names):
        if i < num_metrics:
            metric_importance[i] = weights.get(name, 0.0)
    
    metric_order = np.argsort(metric_importance)[::-1]  # Most important first
    sorted_by_metric = data[:, metric_order]
    
    # Calculate document relevance (weighted sum)
    doc_relevance = np.zeros(num_docs)
    for i in range(min(len(metric_importance), num_metrics)):
        doc_relevance += metric_importance[i] * sorted_by_metric[:, i]
    
    # Sort documents by relevance
    doc_order = np.argsort(doc_relevance)[::-1]  # Most relevant first
    sorted_data = sorted_by_metric[doc_order]
    
    return sorted_data, metric_order, doc_order

# %% [code]
def create_visual_policy_map_animation():
    """Create the complete animation demonstrating Zero-Model Intelligence"""
    # Create figure with custom layout
    fig = plt.figure(figsize=(14, 10))
    gs = gridspec.GridSpec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1])
    ax_main = fig.add_subplot(gs[:, 0])
    ax_zoom = fig.add_subplot(gs[0, 1])
    ax_decision = fig.add_subplot(gs[1, 1])
    
    # Initialize animation state
    current_level = 0
    current_num_docs = NUM_DOCUMENTS
    current_num_metrics = NUM_METRICS
    current_task = TASK
    current_data = generate_policy_data(current_num_docs, current_num_metrics)
    sorted_data, metric_order, doc_order = sort_policy_data(current_data, current_task)
    
    # Animation state variables
    state = "random"  # random -> sorting -> zoom -> decision -> repeat
    frame_count = 0
    total_frames = 0
    
    # Status and explanation text
    status_text = ax_main.text(0.5, -0.1, "", 
                              ha='center', transform=ax_main.transAxes,
                              fontsize=12, color='gray')
    explanation_text = ax_main.text(0.5, -0.15, "",
                                  ha='center', transform=ax_main.transAxes,
                                  fontsize=10, color='gray', style='italic')
    
    # Initial plot setup
    im_main = ax_main.imshow(current_data, cmap='viridis', interpolation='nearest', vmin=0, vmax=1)
    im_zoom = ax_zoom.imshow(np.zeros((5, 5)), cmap='viridis', interpolation='nearest', vmin=0, vmax=1)
    im_decision = ax_decision.imshow(np.zeros((5, 5)), cmap='viridis', interpolation='nearest', vmin=0, vmax=1)
    
    # Configure plots
    for ax in [ax_main, ax_zoom, ax_decision]:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    
    # Add grid lines for better visualization
    ax_main.grid(True, color='white', linestyle='-', linewidth=0.3, alpha=0.2)
    
    # Add colorbar
    cbar = fig.colorbar(im_main, ax=ax_main, shrink=0.7)
    cbar.set_label('Relevance Score (0-1)')
    
    def update(frame):
        """Update function for animation frames"""
        nonlocal current_level, current_num_docs, current_num_metrics
        nonlocal current_task, current_data, sorted_data
        nonlocal state, frame_count, total_frames
        
        # Update state if needed
        if frame_count >= FRAMES_PER_STEP:
            frame_count = 0
            if state == "random":
                state = "sorting"
            elif state == "sorting":
                state = "zoom"
            elif state == "zoom":
                state = "decision"
            elif state == "decision":
                # Move to next level or reset
                if current_level < NUM_LEVELS - 1:
                    current_level += 1
                    # Focus on top documents and most important metrics
                    current_num_docs = max(5, current_num_docs // ZOOM_FACTOR)
                    current_num_metrics = max(3, current_num_metrics // ZOOM_FACTOR)
                    
                    # Generate more focused data
                    current_data = generate_policy_data(current_num_docs, current_num_metrics)
                    sorted_data, _, _ = sort_policy_data(current_data, current_task)
                else:
                    # Reset to beginning for loop
                    current_level = 0
                    current_num_docs = NUM_DOCUMENTS
                    current_num_metrics = NUM_METRICS
                    current_task = TASK
                    current_data = generate_policy_data(current_num_docs, current_num_metrics)
                    sorted_data, _, _ = sort_policy_data(current_data, current_task)
                
                state = "random"
        
        frame_count += 1
        total_frames += 1
        
        # Update plot based on animation state
        if state == "random":
            # Show random data with gradual reveal
            alpha = frame_count / FRAMES_PER_STEP
            display_data = current_data * alpha
            
            im_main.set_array(display_data)
            
            # Update titles and labels
            ax_main.set_title(f"Level {current_level+1}: Raw Policy Data ({current_num_docs} docs × {current_num_metrics} metrics)")
            ax_main.set_xlabel("Metrics (unsorted)")
            ax_main.set_ylabel("Documents (unsorted)")
            
            status_text.set_text("Step 1: Raw policy evaluation data")
            explanation_text.set_text("Each row = document, each column = metric score (0-1)")
            
        elif state == "sorting":
            # Gradually sort the data
            progress = frame_count / FRAMES_PER_STEP
            
            # Interpolate between random and sorted data
            display_data = current_data * (1 - progress) + sorted_data * progress
            
            im_main.set_array(display_data)
            
            # Update titles and labels
            ax_main.set_title(f"Level {current_level+1}: Task-Aware Sorting ({current_num_docs} docs × {current_num_metrics} metrics)")
            ax_main.set_xlabel("Metrics (sorted by importance)")
            ax_main.set_ylabel("Documents (sorted by relevance)")
            
            status_text.set_text("Step 2: Spatial organization encodes task logic")
            explanation_text.set_text("Top-left = most relevant document on most important metric")
            
        elif state == "zoom":
            # Show zoom effect to top-left corner
            progress = frame_count / FRAMES_PER_STEP
            
            # Calculate the zoom region (top-left corner)
            zoom_size_docs = max(5, current_num_docs // ZOOM_FACTOR)
            zoom_size_metrics = max(3, current_num_metrics // ZOOM_FACTOR)
            
            # Highlight the zoom region in main plot
            display_data = np.copy(sorted_data)
            
            # Draw rectangle around zoom region with animation effect
            rect_width = min(int(zoom_size_metrics * (1 - progress*0.7)), current_num_metrics)
            rect_height = min(int(zoom_size_docs * (1 - progress*0.7)), current_num_docs)
            
            # Draw rectangle (thicker as progress increases)
            thickness = max(1, int(3 * progress))
            
            # Only draw if we have enough space
            if rect_width > 0 and rect_height > 0:
                # Top edge
                if 0 < thickness <= current_num_docs and 0 < rect_width <= current_num_metrics:
                    display_data[0:thickness, 0:rect_width] = 1.0
                # Left edge
                if 0 < rect_height <= current_num_docs and 0 < thickness <= current_num_metrics:
                    display_data[0:rect_height, 0:thickness] = 1.0
                # Bottom edge
                if 0 < rect_height-thickness < rect_height <= current_num_docs and 0 < rect_width <= current_num_metrics:
                    display_data[rect_height-thickness:rect_height, 0:rect_width] = 1.0
                # Right edge
                if 0 < rect_height <= current_num_docs and 0 < rect_width-thickness < rect_width <= current_num_metrics:
                    display_data[0:rect_height, rect_width-thickness:rect_width] = 1.0
            
            im_main.set_array(display_data)
            
            # Update zoom plot
            zoom_data = sorted_data[:zoom_size_docs, :zoom_size_metrics]
            im_zoom.set_array(zoom_data)
            
            # Update titles
            ax_main.set_title(f"Level {current_level+1}: Focusing on Relevant Region")
            ax_zoom.set_title(f"Zoomed Region ({zoom_size_docs}×{zoom_size_metrics})")
            ax_zoom.set_xlabel("Top Metrics")
            ax_zoom.set_ylabel("Top Documents")
            
            status_text.set_text("Step 3: Focusing on most relevant region")
            explanation_text.set_text(f"Top {zoom_size_docs} documents on top {zoom_size_metrics} metrics")
            
        elif state == "decision":
            # Show decision process
            progress = frame_count / FRAMES_PER_STEP
            
            # Flash the top-left pixel to simulate "click"
            display_data = np.copy(sorted_data)
            
            # Highlight the decision point (only if we have data)
            if current_num_docs > 0 and current_num_metrics > 0:
                if int(frame_count % 4) < 2:  # Flash effect
                    display_data[0, 0] = 1.2  # Highlight top-left
            
            im_main.set_array(display_data)
            
            # Show decision in the decision plot
            decision_data = np.zeros((5, 5))
            
            # Draw an arrow pointing to top-left
            for i in range(5):
                for j in range(5):
                    if i <= j:
                        decision_data[i, j] = 0.7
            # Highlight the decision point
            decision_data[0, 4] = 1.0
            
            im_decision.set_array(decision_data)
            
            # Update titles
            ax_main.set_title(f"Level {current_level+1}: Decision Point Identified")
            ax_decision.set_title("Edge Device Decision Process")
            
            status_text.set_text("Step 4: Zero-Model Intelligence in action")
            explanation_text.set_text("Edge device checks top-left pixel (180 bytes of code)")
        
        # Update color limits for proper visualization
        im_main.set_clim(0, 1.2)  # Allow for highlight
        im_zoom.set_clim(0, 1)
        im_decision.set_clim(0, 1)
        
        return [im_main, im_zoom, im_decision, status_text, explanation_text]
    
    # Create animation
    total_frames = FRAMES_PER_STEP * 4 * (NUM_LEVELS + 1)
    ani = FuncAnimation(fig, update, frames=total_frames,
                       interval=1000/FPS, blit=True)
    
    return ani, fig

# %% [code]
# Generate and save the animation
print("Generating Visual Policy Map animation (this may take 1-2 minutes)...")
start_time = time.time()

# Create the animation
animation, fig = create_visual_policy_map_animation()

# Save animation as GIF
output_path = 'visual_policy_map_demo.gif'
animation.save(output_path, writer='pillow', fps=FPS)
print(f"Animation saved to {output_path}")
print(f"Generation time: {time.time() - start_time:.2f} seconds")

# Display the GIF in the notebook
display(Image(filename=output_path))

# %% [markdown]
# # Understanding the Animation: The Medium *IS* the Message
# 
# This animation demonstrates the core concept of Zero-Model Intelligence (ZeroMI): **the intelligence isn't in the processing—it's in the data structure itself.**
# 
# ## Key Stages in the Animation
# 
# 1. **Raw Policy Data (Random State)**:
#    - Shows unsorted policy evaluation data (80 documents × 40 metrics)
#    - Each cell represents a score (0-1) for a document on a specific metric
#    - Data appears as a jumbled grid with no clear pattern
# 
# 2. **Task-Aware Sorting**:
#    - Documents are sorted vertically by relevance to the current task ("uncertainty")
#    - Metrics are sorted horizontally by importance to the task
#    - The most relevant information concentrates in the **top-left corner**
#    - This transforms the grid from passive storage to an **active decision map**
# 
# 3. **Focusing on Relevant Region**:
#    - A rectangle highlights the most relevant section (top-left)
#    - The system zooms into this region for closer examination
#    - This demonstrates how the spatial organization guides attention
# 
# 4. **Decision Process**:
#    - The edge device needs only check the **top-left pixel value**
#    - No complex model needed—just "is this pixel dark enough?"
#    - This is Zero-Model Intelligence in action (180 bytes of code)
# 
# 5. **Hierarchical Zooming**:
#    - The process repeats at multiple levels of detail
#    - Each level provides more granular decision-making
#    - Demonstrates how ZeroMI scales from coarse to fine decisions
# 
# ## Why This Matters
# 
# This animation visualizes Marshall McLuhan's insight that **"the medium is the message"** in the context of AI decision-making:
# 
# - Traditional AI: Medium = passive container, Message = raw data
# - ZeroMI: Medium = spatially-organized map, Message = decision logic
# 
# The spatial organization **is** the intelligence—it tells the AI exactly where to look without requiring complex processing at decision time. This enables:
# - **Zero-model intelligence** on devices with <25KB memory
# - **Sub-millisecond decisions** through spatial lookup
# - **Human-AI alignment** through shared cognitive ground
# 
# ## Edge Device Code (180 bytes)
# 
# The animation shows how an edge device with extreme memory constraints can make intelligent decisions:
# 
# ```lua
# -- 180 bytes of code - works on 25KB memory devices
# function process_tile(tile_data)
#     -- Parse tile: [width, height, x, y, pixels...]
#     local width = string.byte(tile_data, 1)
#     local height = string.byte(tile_data, 2)
#     local